In [1]:
import os
import re
import pandas as pd

### IEMOCAP extract

In [ ]:
# Base directory of IEMOCAP dataset
base_dir = "../IEMOCAP_full_release"

# Define emotion mapping
emotion_mapping = {
    "fru": "frustration",
    "sad": "sadness",
    "hap": "happiness",
    "ang": "anger",
    "neu": "neutral"
}

# MELD CSV structure
columns = ["Dialogue_ID", "Utterance_ID", "Speaker", "Emotion", "Utterance"]

In [3]:
# Function to extract emotions from EmoEvaluation files
def parse_emotion_annotations(emotion_file):
    emotion_dict = {}
    with open(emotion_file, "r") as f:
        for line in f:
            match = re.match(r"\[(.*?)\]\s+(\S+)\s+(\S+)\s+\[.*?\]", line)
            if match:
                start_end, utterance_id, emotion = match.groups()
                if emotion in emotion_mapping:
                    emotion_dict[utterance_id] = emotion_mapping[emotion]
    return emotion_dict

In [7]:
# Function to process one session
def process_session(session, dialogue_counter):
    session_path = os.path.join(base_dir, session)
    dialog_path = os.path.join(session_path, "dialog")
    transcripts_path = os.path.join(dialog_path, "transcriptions")
    emo_eval_path = os.path.join(dialog_path, "EmoEvaluation")

    session_data = []

    # Process each transcript file
    for transcript_file in sorted(os.listdir(transcripts_path)):
        if not transcript_file.endswith(".txt"):
            continue  # Skip non-text files

        dialogue_id = dialogue_counter
        dialogue_counter += 1

        # Read the transcript
        with open(os.path.join(transcripts_path, transcript_file), "r") as f:
            transcript_lines = f.readlines()

        # Find corresponding emotion annotation file
        emotion_file = os.path.join(emo_eval_path, transcript_file)
        if not os.path.exists(emotion_file):
            continue  # Skip if corresponding emotion file not found

        # Parse emotion annotations
        emotion_labels = parse_emotion_annotations(emotion_file)

        utterance_counter = 0  # Track position within dialogue

        # Process each line in transcript
        for line in transcript_lines:
            match = re.match(r"(\S+)\s+\[.*?\]:\s+(.*)", line)
            if match:
                utterance_id, text = match.groups()
                speaker = utterance_id[:-3]  # Extract speaker id
                emotion = emotion_labels.get(utterance_id, "neutral")  # Default to neutral if missing

                session_data.append([dialogue_id, utterance_counter, speaker, emotion, text])
                utterance_counter += 1

    return session_data, dialogue_counter

In [8]:
# Process all five sessions
all_data = []
dialogue_counter = 0

for session in ["session1", "session2", "session3", "session4", "session5"]:
    session_data, dialogue_counter = process_session(session, dialogue_counter)
    all_data.extend(session_data)

In [ ]:
# Convert to DataFrame and save as CSV
df = pd.DataFrame(all_data, columns=columns)
df.to_csv("./iemocap.csv", index=False)
print("Conversion completed. Data saved as 'iemocap.csv'.")

Conversion completed. Data saved as 'iemocap.csv'.


### IEMOCAP data engineering

there are too few dialogues in iemocap.
need to split them up into more dialogues.
speaker will be preserved and emotionflow can learn global speaker features.

In [2]:
# Load dataset
file_path = "./iemocap.csv"
df = pd.read_csv(file_path)

df

,Dialogue_ID,Utterance_ID,Speaker,Emotion,Utterance
0,0,0,Ses01F_impro01_F,neutral,Excuse me.
1,0,1,Ses01F_impro01_M,frustration,Do you have your forms?
2,0,2,Ses01F_impro01_F,neutral,Yeah.
3,0,3,Ses01F_impro01_M,frustration,Let me see them.
4,0,4,Ses01F_impro01_F,neutral,Is there a problem?
...,...,...,...,...,...
10082,150,86,Ses05M_script03_2_M,anger,oh! Marry you again? I wouldn't marry you agai...
10083,150,87,Ses05M_script03_2_F,anger,Beast
10084,150,88,Ses05M_script03_2_M,anger,You're a wicked little vampire. And I pray to...
10085,150,89,Ses05M_script03_2_F,anger,Brute


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10087 entries, 0 to 10086
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Dialogue_ID   10087 non-null  int64 
 1   Utterance_ID  10087 non-null  int64 
 2   Speaker       10087 non-null  object
 3   Emotion       10087 non-null  object
 4   Utterance     10086 non-null  object
dtypes: int64(2), object(3)
memory usage: 394.1+ KB


In [7]:
min_utterances = 8
new_rows = []
new_dialogue_id = 0

for _, group in df.groupby("Dialogue_ID"):
    group = group.sort_values("Utterance_ID").reset_index(drop=True)
    total_utts = len(group)

    # Calculate max number of splits we can make with min_utterances per split
    max_splits = total_utts // min_utterances
    n_splits = min(10, max_splits)
    
    if n_splits == 0:
        # If not even 1 chunk of 8 is possible, just keep it as one
        chunk = group.copy()
        chunk["Dialogue_ID"] = new_dialogue_id
        chunk["Utterance_ID"] = range(len(chunk))
        new_rows.append(chunk)
        new_dialogue_id += 1
        continue

    # Distribute utterances as evenly as possible across n_splits
    sizes = [total_utts // n_splits] * n_splits
    for i in range(total_utts % n_splits):
        sizes[i] += 1

    start = 0
    for size in sizes:
        chunk = group.iloc[start:start+size].copy()
        chunk["Dialogue_ID"] = new_dialogue_id
        chunk["Utterance_ID"] = range(len(chunk))
        new_rows.append(chunk)
        new_dialogue_id += 1
        start += size

# Combine all mini-dialogues
new_df = pd.concat(new_rows, ignore_index=True)

In [8]:
print(new_df["Dialogue_ID"].nunique())
print(new_df.groupby("Dialogue_ID")["Utterance_ID"].max().head())

1141
Dialogue_ID
0    9
1    9
2    9
3    9
4    9
Name: Utterance_ID, dtype: int64


In [9]:
new_df

,Dialogue_ID,Utterance_ID,Speaker,Emotion,Utterance
0,0,0,Ses01F_impro01_F,neutral,Excuse me.
1,0,1,Ses01F_impro01_M,frustration,Do you have your forms?
2,0,2,Ses01F_impro01_F,neutral,Yeah.
3,0,3,Ses01F_impro01_M,frustration,Let me see them.
4,0,4,Ses01F_impro01_F,neutral,Is there a problem?
...,...,...,...,...,...
10082,1140,4,Ses05M_script03_2_M,anger,oh! Marry you again? I wouldn't marry you agai...
10083,1140,5,Ses05M_script03_2_F,anger,Beast
10084,1140,6,Ses05M_script03_2_M,anger,You're a wicked little vampire. And I pray to...
10085,1140,7,Ses05M_script03_2_F,anger,Brute


In [10]:
new_df.to_csv("./iemocap.csv", index=False)